In [1]:
import datetime
from datetime import datetime as dt

from elasticsearch import Elasticsearch
from django.conf import settings
from pprint import pprint

# from literev.libs.parsing import process_search_query_elasticsearchIn

In [2]:
es = Elasticsearch(
            [settings.ES_HOST_URL],
            basic_auth=(settings.ES_USERNAME, settings.ES_PASSWORD),
        )
es.cluster.health()

ObjectApiResponse({'cluster_name': 'docker-cluster', 'status': 'yellow', 'timed_out': False, 'number_of_nodes': 1, 'number_of_data_nodes': 1, 'active_primary_shards': 9, 'active_shards': 9, 'relocating_shards': 0, 'initializing_shards': 0, 'unassigned_shards': 3, 'delayed_unassigned_shards': 0, 'number_of_pending_tasks': 0, 'number_of_in_flight_fetch': 0, 'task_max_waiting_in_queue_millis': 0, 'active_shards_percent_as_number': 75.0})

### Create a function that queries Elasticsearch and returns specific fields:
    (
        procedure_type, 
        decision_type, 
        decision_date, 
        descriptors, 
        result, 
        standards
        
    ) 

In [12]:
# Querying by keyword and date based on the decision date.

In [15]:

def query_elasticsearch(index_name: str, query: str, start_date: datetime.date, end_date: datetime.date):
    """
    Queries Elasticsearch and returns specific fields from the documents.

    Args:
        es_host (str): The Elasticsearch host.
        index_name (str): The name of the Elasticsearch index.
        query (str): The query to run against the index.

    Returns:
        list[dict]: A list of dictionaries, each containing the specified fields from a document.
    """
    es = Elasticsearch(
            [settings.ES_HOST_URL],
            basic_auth=(settings.ES_USERNAME, settings.ES_PASSWORD),
        )

    es_query = {
        "query": {
            "bool": {
                "must": [
                    {
                        "multi_match": {
                            "query": query,
                            "fields": ["document_text"],
                            "slop": 999,
                        }
                    }
                ],
                "filter": [
                    {
                        "range": {
                            "decision_date": {
                                "gte": start_date.strftime(
                                    "%Y-%m-%d"
                                ),
                                "lte": end_date.strftime("%Y-%m-%d"),
                            }
                        }
                    }
                ],
                }
            }
        }
    
    response = es.search(index=index_name, body=es_query)
    hits = response['hits']['hits']
    char_limit=1200

    # Extract the specified fields from each hit
    extracted_data = [
        {
            'procedure_type': hit['_source'].get('procedure_type', ''),
            'decision_type': hit['_source'].get('decision_type', ''),
            'decision_date': hit['_source'].get('decision_date', ''),
            'descriptors': hit['_source'].get('descriptors', ''),
            'result': hit['_source'].get('result', ''),
            'standards': hit['_source'].get('standards', ''),
            'summary': hit['_source'].get("summary", ""),
            'document_text': (hit['_source'].get("document_text", "")[:char_limit] + hit['_source'].get("document_text", "")[char_limit:].split('.', 1)[0] + '.') if hit['_source'].get("document_text") else "",
        }
        for hit in hits
    ]

    return extracted_data


In [20]:
ES_JUDICIARY_INDEX_NAME = "judiciary"
ES_QUERY = 'divorce'
date_begin = datetime.date(2020, 1, 1)
date_end = datetime.date(2021, 1, 1)

results = query_elasticsearch(ES_JUDICIARY_INDEX_NAME, ES_QUERY, date_begin, date_end)
for result in results:
    pprint(result)

{'decision_date': '2020-04-16',
 'decision_type': 'ATA/363/2020',
 'descriptors': "BOURSE D'ÉTUDES;CHAMP D'APPLICATION(EN "
                'GÉNÉRAL);CONCLUSIONS;JUGEMENT DE DIVORCE;OBLIGATION '
                "D'ENTRETIEN;MAJORITÉ(ÂGE)",
 'document_text': '     république et\n'
                  '    canton de genève\n'
                  '   \xa0  pouvoir judiciaire\n'
                  'A/2723/2019-FORMA ATA/363/2020 \n'
                  'COUR DE JUSTICE\n'
                  'Chambre administrative \n'
                  'Arrêt du 16 avril 2020\n'
                  '2ème section\n'
                  ' \xa0           dans la cause\n'
                  '\xa0\n'
                  'M. A______ \n'
                  'contre\n'
                  "SERVICE DES BOURSES ET PRÊTS D'ÉTUDES \n"
                  '\xa0\n'
                  '      EN FAIT\n'
                  '1) Le 31 janvier 2019, M. A______, né le ______ 1994, a '
                  'déposé une demande de renouvellement de bourse

### Collectors

In [21]:
# collectors.py
from __future__ import annotations
import datetime
from dataclasses import dataclass
from django.conf import settings
from elasticsearch import Elasticsearch
from literev.libs.parsing import process_search_query_elasticsearch

@dataclass
class MetaData:
    doc_id: str
    document_text: str
    procedure_type: str
    decision_type: str
    decision_date: str
    descriptors: str
    summary: str
    standards: str
    result: str


class ElasticSearchCollector:
    """Implements all essential methods for collecting data from elasticsearch sources."""

    es: Elasticsearch
    ES_PAGE_SIZE: int = 1000
    ES_INDEX_NAME: str = "judiciary"

    def __init__(self) -> None:
        self.es = Elasticsearch(
            [settings.ES_HOST_URL],
            basic_auth=(settings.ES_USERNAME, settings.ES_PASSWORD),
        )

    def collect_documents(
        self, search: str, date_begin: datetime.date, date_end: datetime.date
    ) -> list[MetaData]:
        """Retrieve articles based on the provided search parameters."""

        es_query = process_search_query_elasticsearch(
            search_query=search,
            start_date=date_begin,
            end_date=date_end,
        )

        # get all documents from every page response from elasticsearch
        documents = self.get_all_documents_from_es_response(es_query)

        result = []

        for doc in documents:
            metadata = self.extract_document_metadata(doc)

            if metadata:
                result.append(metadata)
            else:
                print(
                    f"This document does not have document_text field: {doc}"
                )

        return result

    def get_all_documents_from_es_response(
        self,
        es_query: dict[str, int | list[str] | dict[str, str]],
    ) -> list[dict[str, str]]:
        """Get all articles from elasticsearch response."""

        es_query["size"] = self.ES_PAGE_SIZE

        response = self.es.search(
            index=self.ES_INDEX_NAME, body=es_query, scroll="2m"
        )

        scroll_id = response["_scroll_id"]
        hits = response["hits"]["hits"]

        documents = []

        # process the first page from elasticsearch
        documents += self._process_documents_from_es_response_page(hits)

        # then we process the rest of the pages if they exist
        # by passing the scroll_id to es.scroll
        while hits:
            response = self.es.scroll(scroll_id=scroll_id, scroll="2m")
            hits = response["hits"]["hits"]
            documents += self._process_documents_from_es_response_page(hits)

        return documents

    def _process_documents_from_es_response_page(
        self, hits: list[dict[str, dict[str, str]]]
    ) -> list[dict[str, str]]:
        """Process all articles from elasticsearch response page."""
        documents = []
        for es_hit in hits:
            # get article from elasticsearch hit _source key
            article = es_hit["_source"]
            if article:
                documents.append(article)
        return documents

    def extract_document_metadata(
        self, document: dict[str, str]
    ) -> MetaData | None:
        """Create Metadata from source article."""
        # TODO: Implement char_limit only in the plot
        char_limit = 200
        
        doc_id = document.get("id")
        document_text = document.get("document_text", "")[:char_limit]
        procedure_type = document.get("procedure_type", "")
        decision_type = document.get("decision_type", "")
        decision_date = document.get("decision_date", "")
        descriptors = document.get("descriptors", "")
        summary = document.get("summary", "")
        standards = document.get("standards", "")
        result = document.get("result", "")

        if document_text:
            metadata = MetaData(
                doc_id=doc_id,
                document_text=document_text,
                procedure_type=procedure_type,
                decision_type=decision_type,
                decision_date=decision_date,
                descriptors=descriptors,
                summary=summary,
                standards=standards,
                result=result,
            )

            return metadata

        return None

    def get_max_documents(
        self, search: str, begin: datetime.date, end: datetime.date
    ) -> int:
        """Counts total number of articles for a given query."""
        es_query = process_search_query_elasticsearch(
            search_query=search,
            start_date=begin,
            end_date=end,
        )

        response = self.es.count(index=self.ES_INDEX_NAME, body=es_query)

        return int(response["count"])

    def create_document_from_metadata(self, metadata: MetaData) -> None:
        """Create document from metadata."""
        pass


#################
query = "divorce"
date_begin = datetime.date(2020, 1, 1)
date_end = datetime.date(2021, 1, 1)
metadata = ElasticSearchCollector().collect_documents(query, date_begin, date_end)
#####################################

In [22]:
print(len(metadata))
from pprint import pprint
pprint(metadata)

58
[MetaData(doc_id='1588572672',
          document_text='     république et\n'
                        '    canton de genève\n'
                        '   \xa0  pouvoir judiciaire\n'
                        'A/2723/2019-FORMA ATA/363/2020 \n'
                        'COUR DE JUSTICE\n'
                        'Chambre administrative \n'
                        'Arrêt du 16 avril 2020\n'
                        '2ème section\n'
                        ' \xa0           dans la cause\n',
          procedure_type='A/2723/2019',
          decision_type='ATA/363/2020',
          decision_date='2020-04-16',
          descriptors="BOURSE D'ÉTUDES;CHAMP D'APPLICATION(EN "
                      'GÉNÉRAL);CONCLUSIONS;JUGEMENT DE DIVORCE;OBLIGATION '
                      "D'ENTRETIEN;MAJORITÉ(ÂGE)",
          summary='Le jugement de divorce des parents du recourant ne prévoit '
                  "pas de contribution d'entretien en faveur de l'intéressé "
                  "au-delà de sa majori

---